In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [3]:
device = torch.device("cpu")

In [4]:
latent_dim = 100
lr = 0.001
epochs = 20
batch_size = 64

In [12]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])
])

In [13]:
dataloader = DataLoader(datasets.MNIST(root="./data", train=True, transform=transform, download=True), shuffle=True, batch_size=batch_size)

In [16]:
class Generator(nn.Module):
  def __init__(self):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(latent_dim,256),
        nn.ReLU(),

        nn.Linear(256, 512),
        nn.ReLU(),

        nn.Linear(512,1024),
        nn.ReLU(),

        nn.Linear(1024,28*28),
        nn.Tanh()
    )

  def forward(self, x):
    return self.model(x).view(-1,1,28,28)

class Discriminator(nn.Module):
  def __init__(self):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(28*28,512),
        nn.LeakyReLU(0.2),

        nn.Linear(512,256),
        nn.LeakyReLU(),

        nn.Linear(256,1),
        nn.Sigmoid()
    )

  def forward(self, x):
    return self.model(x.view(-1,28*28))


In [17]:
G = Generator().to(device)
D = Discriminator().to(device)
criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=lr)
optimizer_D = optim.Adam(D.parameters(), lr=lr)

In [ ]:
for epoch in range(epochs):
  for image, _ in dataloader:
    image = image.to(device)
    batch_size = image.size(0)

    real_labels = torch.ones(batch_size,1).to(device)
    fake_labels = torch.zeros(batch_size,1).to(device)

    z = torch.randn(batch_size, latent_dim).to(device)
    fake_img = G(z)

    real_loss = criterion(D(image), real_labels)
    fake_loss = criterion(D(fake_img.detach()),fake_labels)
    d_loss = real_loss + fake_loss

    optimizer_D.zero_grad()
    d_loss.backward()
    optimizer_D.step()

    g_loss = criterion(D(fake_img), real_labels)

    optimizer_G.zero_grad()
    g_loss.backward()
    optimizer_G.step()

  print(f"Epoch:{epoch+1}, D Loss:{d_loss.item():.4f}, G Loss:{g_loss.item():.4f}")

Epoch:1, D Loss:100.0000, G Loss:100.0000
Epoch:2, D Loss:100.0000, G Loss:100.0000
Epoch:3, D Loss:100.0000, G Loss:100.0000
Epoch:4, D Loss:100.0000, G Loss:100.0000
Epoch:5, D Loss:100.0000, G Loss:100.0000
Epoch:6, D Loss:100.0000, G Loss:100.0000
Epoch:7, D Loss:100.0000, G Loss:100.0000
Epoch:8, D Loss:100.0000, G Loss:100.0000
Epoch:9, D Loss:100.0000, G Loss:100.0000
Epoch:10, D Loss:100.0000, G Loss:100.0000
Epoch:11, D Loss:100.0000, G Loss:100.0000
Epoch:12, D Loss:100.0000, G Loss:100.0000
Epoch:13, D Loss:100.0000, G Loss:100.0000
Epoch:14, D Loss:100.0000, G Loss:100.0000
Epoch:15, D Loss:100.0000, G Loss:100.0000
Epoch:16, D Loss:100.0000, G Loss:100.0000
Epoch:17, D Loss:100.0000, G Loss:100.0000


In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z)

# Discriminator (CNN)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),

            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x).view(-1, 1)

# Initialize models
G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

In [ ]:
 # ---- Save sample images every epoch ----
    with torch.no_grad():
        sample_z = torch.randn(25, latent_dim).to(device)
        sample_imgs = (G(sample_z) + 1) / 2
        save_image(sample_imgs, f"images/epoch_{epoch+1}.png", nrow=5)

# ------------------ Save Model ------------------
torch.save(G.state_dict(), "generator.pth")

# ------------------ Generate Final Images ------------------
G.eval()
with torch.no_grad():
    z = torch.randn(25, latent_dim).to(device)
    final_imgs = (G(z) + 1) / 2
    save_image(final_imgs, "images/final.png", nrow=5)

print("Training complete. Images saved in 'images/' folder.")